# Backpropagation on XOR

Train a small network on XOR and track loss and gradient norms to see backprop in action.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys

sys.path.insert(0, '../src')

from neura_command import Sequential, Dense, Sigmoid, SGD, MSELoss

np.random.seed(42)


## 1. XOR Dataset


In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)


## 2. Build the Network


In [ ]:
model = Sequential(
    layers=[
        Dense(2, 4),
        Sigmoid(),
        Dense(4, 1),
        Sigmoid(),
    ],
    optimizer=SGD(learning_rate=0.5),
    loss=MSELoss(),
)


## 3. Manual Training Loop (to capture gradients)


In [ ]:
epochs = 5000
loss_history = []

dense_layers = [layer for layer in model.layers if hasattr(layer, 'gradients')]
grad_history = [[] for _ in dense_layers]

model.train_mode()
for _ in range(epochs):
    preds = model.forward(X, training=True)
    loss = model.loss.compute(y, preds)
    loss_history.append(float(loss))

    grad = model.loss.derivative(y, preds)
    model.backward(grad)

    for i, layer in enumerate(dense_layers):
        grads = layer.gradients()
        grad_norm = float(np.linalg.norm(grads[0])) if grads and grads[0] is not None else 0.0
        grad_history[i].append(grad_norm)

    model.optimizer.step(model.layers)


## 4. Loss and Gradient Evolution


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(loss_history)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')

plt.subplot(1, 2, 2)
for i, grads in enumerate(grad_history):
    plt.plot(grads, label=f'Dense Layer {i + 1}')
plt.title('Gradient Norms')
plt.xlabel('Epoch')
plt.ylabel('L2 Norm')
plt.legend()

plt.tight_layout()
plt.show()


## 5. Decision Boundary


In [ ]:
def plot_decision_boundary(model, X, y):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    preds = model.predict(grid).reshape(xx.shape)

    plt.contourf(xx, yy, preds, levels=[0, 0.5, 1], alpha=0.7, cmap='Spectral')
    plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap='Spectral', edgecolors='k')
    plt.title('XOR Decision Boundary')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.show()

plot_decision_boundary(model, X, y)


## 6. Predictions


In [ ]:
preds = model.predict(X)
print('Predictions:')
print(np.round(preds, 3))
